# Stage 2 — bootstrapping a €STR OIS discount curve

Stage 1 took observed yields and fitted a smooth shape through them. Stage 2 does the opposite: it takes tradeable swap quotes and solves out discount factors **exactly**, one at a time, with no model at all.

This is the curve a rates desk actually discounts with. Since the 2008–2010 shift away from LIBOR discounting, collateralised euro derivatives are discounted at the overnight rate — €STR — so the €STR OIS curve is *the* euro discount curve.

By the end you should be able to:

1. derive why an OIS floating leg is worth exactly `1 - DF(T)`, and see why that makes the bootstrap trivial,
2. bootstrap a curve with pencil and paper for the first two pillars, then let code do the rest,
3. explain what your interpolation choice does to forward rates, and why that is the only place to look,
4. compute a bucketed delta ladder — the risk report the whole trading floor reads.

In [ ]:
import sys, pathlib, datetime as dt
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eurocurve import ecb
from eurocurve.curve import DiscountCurve
from eurocurve.bootstrap import OISSwap, bootstrap_ois, load_quotes, par_rate, reprice_check, spot_date
from eurocurve.daycount import year_fraction, annual_schedule

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

DATA = pathlib.Path.cwd().parent / 'data' / 'eur_ois_sample.csv'

## 1. The instrument

A EUR OIS: you pay a fixed rate `S`, you receive €STR compounded daily, over the same period. Conventions:

| | |
|---|---|
| index | €STR (euro short-term rate, published by the ECB each morning for the previous day) |
| fixed frequency | annual beyond 1Y; single payment at maturity for 1Y and shorter |
| day count | ACT/360, both legs |
| start | spot, T+2 business days |
| roll | modified following, TARGET2 calendar |

One honest caveat before we start: **live OIS quotes are not free.** They live behind Bloomberg / Refinitiv / ICAP. So `data/eur_ois_sample.csv` is a static, plausible EUR curve bundled with the repo, which makes this notebook reproducible offline. The €STR fixing itself *is* free, so we pull that live and use it to anchor the overnight end. The bootstrap arithmetic is identical whatever numbers you feed it — swap in a real quote file and nothing downstream changes.

In [ ]:
valuation = dt.date.today()
spot = spot_date(valuation)
print('valuation', valuation, '| spot (T+2)', spot)

estr_date, estr = ecb.fetch_estr()
print(f'€STR {estr:.4%} as of {estr_date}  (live from the ECB)')

swaps = load_quotes(DATA, spot=spot)
pd.DataFrame({'tenor': [s.tenor for s in swaps],
              'rate_%': [s.rate*100 for s in swaps],
              'maturity_y': [round(s.maturity, 3) for s in swaps],
              'n_payments': [len(s.pay_dates) for s in swaps]})

### A detour on schedules that costs people real money

Look at the 18M swap. It has **two** payments. Where are they?

In [ ]:
s18 = next(s for s in swaps if s.tenor == '18M')
for d, a in zip(s18.pay_dates, s18.alphas):
    print(f'{d}   ~{(d-spot).days/365:.2f}y after spot   accrual {a:.6f}')

**6M and 18M — not 12M and 24M.** Schedules are generated *backwards from maturity*, because maturity is the date both counterparties agreed on and the stub goes at the front. Generate forwards from the start date instead and you get a plausible-looking curve that is quietly wrong. `tests/test_daycount.py::test_broken_tenor_generates_backwards` exists specifically to catch this.

## 2. Why the floating leg collapses

The swap is worth zero at inception, so `PV(fixed) = PV(float)`:

```
S * sum_i alpha_i * DF(t_i)  =  PV(float)
```

The left side is the fixed rate times the **annuity**. Now the right side. Take one floating period. In a single-curve world — where the rate you project is the same rate you discount with, which for €STR OIS is exactly true — the forward compounded rate over `[t_{i-1}, t_i]` satisfies

```
1 + alpha_i * F_i = DF(t_{i-1}) / DF(t_i)
```

so the PV of that one payment is

```
alpha_i * F_i * DF(t_i) = (DF(t_{i-1})/DF(t_i) - 1) * DF(t_i) = DF(t_{i-1}) - DF(t_i)
```

Sum over all periods. Every interior term appears once positive and once negative. It **telescopes**:

```
PV(float) = DF(t_0) - DF(t_n) = 1 - DF(T)
```

The floating leg is worth "a euro now minus a euro at maturity", whatever the forwards do. Which hands us the par rate:

```
S(T) = (1 - DF(T)) / sum_i alpha_i * DF(t_i)
```

Let us check the telescoping numerically rather than take it on faith — on an arbitrary made-up curve, so we know it is not an artefact of our own construction.

In [ ]:
rng = np.random.default_rng(3)
tt = np.arange(1, 11.0)
junk = DiscountCurve.from_zero_rates(tt, 0.02 + rng.normal(0, 0.004, tt.size))

dfs = np.concatenate([[1.0], np.asarray(junk.df(tt))])
alphas = np.full(10, 1.0)
fwds = (dfs[:-1]/dfs[1:] - 1)/alphas          # forward for each period
pv_the_hard_way = float(np.sum(alphas*fwds*dfs[1:]))
pv_the_easy_way = 1.0 - float(junk.df(10.0))

print(f'sum of discounted forward payments : {pv_the_hard_way:.15f}')
print(f'1 - DF(10)                         : {pv_the_easy_way:.15f}')
print(f'difference                         : {pv_the_hard_way - pv_the_easy_way:.2e}')

Identical to machine precision, on a deliberately silly curve. The result is structural, not numerical.

## 3. Bootstrapping by hand

Now walk up the maturities. The 1Y swap has a **single** payment, so its equation has one unknown:

```
S * alpha_1 * DF(T1) = 1 - DF(T1)
      =>  DF(T1) = 1 / (1 + S * alpha_1)
```

The 2Y swap pays at 1Y and 2Y. You already know `DF(T1)`, so again one equation, one unknown:

```
DF(T2) = (1 - S2*alpha_1*DF(T1)) / (1 + S2*alpha_2)
```

That is the whole method — each instrument contributes exactly one new pillar. Let us do the first two with plain arithmetic, then check the library agrees.

In [ ]:
s1 = next(s for s in swaps if s.tenor == '1Y')
s2 = next(s for s in swaps if s.tenor == '2Y')

a1 = s1.alphas[0]
df1 = 1.0 / (1.0 + s1.rate*a1)

b1, b2 = s2.alphas
df2 = (1.0 - s2.rate*b1*df1) / (1.0 + s2.rate*b2)

print(f'by hand : DF(1Y) = {df1:.12f}   DF(2Y) = {df2:.12f}')

lib = bootstrap_ois([s1, s2])
print(f'library : DF(1Y) = {lib.df(s1.maturity):.12f}   DF(2Y) = {lib.df(s2.maturity):.12f}')
print(f'\nimplied 1y zero (annual comp): {lib.zero(s1.maturity, "annual"):.4%}')
print(f'quoted 1y OIS par rate       : {s1.rate:.4%}')
print('the gap is the ACT/360 vs ACT/365 basis, ~1.4% of the rate — not an error')

That last line is worth pausing on. A 2.005% ACT/360 par rate is **not** a 2.005% annual zero rate. Whenever a number comes back slightly off and you cannot see why, the day count is the first thing to check and it is right about half the time.

## 4. Where the closed form breaks — and why we root-solve

The pencil-and-paper recursion works while every intermediate payment date is already a pillar. It stops working the moment quotes have gaps. Our quote set jumps `10Y → 12Y → 15Y → 20Y → 25Y → 30Y`, so the 15Y swap pays at 11y, 13y and 14y where we have nothing.

The fix: do not solve the recursion, solve the *definition*. For each swap, find the `DF(T)` such that the curve reprices the swap at its quote, letting the intermediate discount factors come from interpolating between the last known pillar and the candidate. `scipy.optimize.brentq`, twenty lines, always correct.

This also makes something explicit that is easy to forget: **the interpolation scheme is part of the curve's definition, not a cosmetic step applied afterwards.** Change the interpolation and you change the bootstrapped pillars themselves.

In [ ]:
curve = bootstrap_ois(swaps, overnight_rate=estr, as_of=valuation)
print(curve)

chk = reprice_check(swaps, curve)
print(f"\nworst repricing error: {chk['error_bp'].abs().max():.3e} bp")
chk.round(8)

**Zero to machine precision, on every instrument.** That is the defining property of a bootstrap and the sharpest contrast with Stage 1, where the fitted curve missed the observed yields by a basis point or two.

Run this check after every change you ever make to curve code. It catches day-count errors, schedule errors and percent-vs-decimal errors in a single line, and it costs nothing.

When to want which:

| | bootstrap | parametric fit |
|---|---|---|
| reprices inputs | exactly | approximately |
| degrees of freedom | one per instrument | six, total |
| absorbs quote noise | no — it enshrines it | yes |
| parameters mean anything | no | yes, loosely |
| use it for | marking and hedging a book | comparing curves across time or country |

## 5. The curve in three languages

In [ ]:
probe = np.array([0.25, 0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30], dtype=float)
curve.to_frame(probe).round(6)

In [ ]:
g = np.linspace(0.02, 30, 600)
starts = np.arange(0, 29.01, 0.25)
fwd1y = np.array([curve.forward(s, s+1.0) for s in starts])

fig, ax = plt.subplots(3, 1, figsize=(9, 11), sharex=True)
ax[0].plot(g, curve.zero(g)*100, lw=2, label='zero (continuous)')
ax[0].plot([s.maturity for s in swaps], [s.rate*100 for s in swaps], 'o', ms=5, label='OIS par quotes')
ax[0].set_ylabel('%'); ax[0].legend(); ax[0].set_title('EUR €STR OIS curve')

ax[1].plot(g, curve.df(g), lw=2, color='seagreen')
ax[1].set_ylabel('discount factor')

ax[2].step(starts, fwd1y*100, where='post', color='crimson')
ax[2].set_ylabel('%'); ax[2].set_xlabel('years')
ax[2].set_title('1y forward rate starting in t — where the ECB path lives')
plt.tight_layout()

The bottom panel is the one traders look at. The 1y forward rate starting in t is, near enough, the market's expectation of where the overnight rate will average during that year — that is, the priced-in ECB path. When someone says "the market has two more cuts priced by June", they read it off this line.

## 6. Interpolation: the choice that hides in the forwards

We interpolate linearly in `log(DF)`. Since `f(t) = -d ln DF/dt`, that means **piecewise constant instantaneous forwards** — the most defensible assumption between two pillars, since you learned nothing in between.

The popular alternative, linear on zero rates, looks identical on a zero-curve chart. Watch what it does to forwards.

In [ ]:
pillars = curve.times[1:]
zeros = np.asarray(curve.zero(pillars))

g = np.linspace(0.1, 30, 2000)
zz_linear = np.interp(g, pillars, zeros)          # linear-on-zeros
df_linear = np.exp(-zz_linear*g)
f_linear = -np.gradient(np.log(df_linear), g)
f_loglin = np.asarray(curve.inst_forward(g))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(g, zz_linear*100, label='linear on zeros')
ax[0].plot(g, np.asarray(curve.zero(g))*100, '--', label='log-linear on DF')
ax[0].set_title('zero curves — indistinguishable'); ax[0].legend()
ax[1].plot(g, f_linear*100, label='linear on zeros', alpha=0.8)
ax[1].plot(g, f_loglin*100, '--', label='log-linear on DF', alpha=0.8)
ax[1].set_title('instantaneous forwards — not remotely the same'); ax[1].legend()
for a in ax: a.set_xlabel('years'); a.set_ylabel('%')
plt.tight_layout()

Two schemes, the same pillars, the same repricing of every input instrument, and materially different forward rates in the gaps. If you were pricing a forward-starting swap in the 11y–14y window, the two curves would give you different numbers and both would pass the repricing check.

**The rule: judge an interpolation method by its forward curve. Never by its zero curve.** Every method looks fine on the zero curve. That is exactly why the zero curve is the wrong place to look.

## 7. Govt versus OIS — what the spread means

Now put Stage 1 and Stage 2 side by side. Both are "the euro risk-free curve", built from different instruments, and they do not agree.

In [ ]:
try:
    from eurocurve.nss import fit_nss, nss_spot
    obs = ecb.fetch_curve()
    p, _ = fit_nss(obs['maturity'].values, obs['rate'].values)

    g = np.linspace(0.5, 30, 300)
    govt = np.asarray(nss_spot(g, p))
    ois = np.asarray(curve.zero(g))

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].plot(g, govt*100, label='AAA govt (Stage 1, fitted)')
    ax[0].plot(g, ois*100, label='€STR OIS (Stage 2, bootstrapped)')
    ax[0].legend(); ax[0].set_title('two euro curves'); ax[0].set_ylabel('%')
    ax[1].plot(g, (govt-ois)*1e4, color='crimson')
    ax[1].axhline(0, color='k', lw=0.8)
    ax[1].set_title('govt minus OIS (bp) — the asset swap spread')
    for a in ax: a.set_xlabel('years')
    plt.tight_layout()
except Exception as exc:
    print('needs network for the ECB pull:', exc)

Caveat first: our OIS quotes are synthetic, so the *level* of this spread is not meaningful. The *concept* is.

This gap is roughly the asset swap spread, and it is not a mistake in either curve. Bunds trade rich to OIS for reasons that have nothing to do with credit: they are the deliverable into futures, the preferred collateral in a stress, subject to regulatory demand from banks needing HQLA, and in limited supply. A negative govt-minus-OIS spread means "investors pay a premium to hold the actual bond rather than the swap". It went sharply more negative during the 2022 collateral squeeze and again whenever the market gets nervous.

So: two curves, both correct, measuring different things. Discount collateralised derivatives on OIS. Value a bond portfolio against the govt curve. Do not mix them and then wonder why nothing reconciles.

## 8. Risk — the bucketed delta ladder

The reason anyone builds a curve is to compute risk off it. The standard report: bump each input quote by 1bp, rebuild the whole curve, reprice the trade, record the PV change. That tells you which instruments to trade to hedge — which is the only form of risk anyone can act on.

`OISSwap.pv` is written from the **fixed receiver's** point of view, so a positive delta means the trade gains when that bucket's rate rises. A receiver loses when rates rise, so expect negative numbers.

In [ ]:
NOTIONAL = 1e8   # EUR 100m

# par_rate() only needs a schedule, not a rate, so make a throwaway swap to read
# the fair 10y rate off the curve, then restrike the real trade at it.
fair = par_rate(OISSwap('10Y', 0.0, spot), curve)
trade = OISSwap('10Y', fair, spot)        # RECEIVE fixed, EUR 100m

base_pv = trade.pv(curve, NOTIONAL)
rows = []
for i, s in enumerate(swaps):
    bumped = [OISSwap(x.tenor, x.rate + (1e-4 if j == i else 0.0), spot)
              for j, x in enumerate(swaps)]
    c2 = bootstrap_ois(bumped, overnight_rate=estr)
    rows.append({'bucket': s.tenor, 'delta_per_bp': trade.pv(c2, NOTIONAL) - base_pv})

ladder = pd.DataFrame(rows)
print(f'trade: receive fixed {trade.rate:.4%} on EUR {NOTIONAL:,.0f} for 10y')
print(f'base PV: {base_pv:,.2f}   (should be ~0 -- it is struck at market)')
print(f'total delta: {ladder["delta_per_bp"].sum():,.0f} per 1bp parallel shift')
ladder.round(1)

In [ ]:
plt.bar(range(len(ladder)), ladder['delta_per_bp'])
plt.xticks(range(len(ladder)), ladder['bucket'], rotation=45)
plt.axhline(0, color='k', lw=0.8)
plt.ylabel('PV change per 1bp'); plt.title('bucketed delta, 10y receiver on EUR 100m')
plt.tight_layout(); plt.show()

Almost all the risk sits in the 10Y bucket, which is exactly right — bumping the 10Y quote is the only bump that moves the pillar this trade depends on most. The small amounts leaking into neighbouring buckets are the interpolation redistributing things, and the fact that they are small is a sign the curve is well behaved. A curve construction that smears one instrument's risk across five buckets is one you cannot hedge cleanly, and that is a legitimate reason to reject an interpolation scheme.

Sanity check on magnitude: the delta of an at-market swap is roughly `notional × annuity × 1bp`. The annuity of a 10y annual EUR swap around 2.6% is a bit under 9, so expect something in the region of **EUR 85,000–90,000 per basis point**, negative for a receiver. If your number is ten times off, you have a notional or a day-count problem, not a curve problem.

## Exercises

1. **Break the day count.** Change `ACCRUAL_BASIS` in `bootstrap.py` from ACT/360 to ACT/365F, rerun. The repricing check still passes perfectly — the curve is internally consistent, just wrong. Now compute the PV of a 30y swap under both. How many euros per EUR 100m did that one string cost? This is the lesson: internal consistency is not correctness.

2. **Rebuild with fewer quotes.** Bootstrap using only 1Y, 5Y, 10Y, 30Y. Compare the resulting 7y zero rate to the full curve's. How much does a pillar cost you when it is missing?

3. **Implement linear-on-zeros bootstrapping properly.** Not as a post-hoc interpolation of the pillars, but inside the root solve, so the pillars themselves change. Do the bootstrapped discount factors differ? By how much at 12y?

4. **Forward-starting swap.** Price a 5y swap starting in 5 years off this curve. Then check it against the identity that a 5y5y forward swap plus a spot 5y swap replicates a spot 10y swap — annuity-weighted, not by simple addition. Getting that weighting right is a real interview question.

5. **Invert the exercise.** Take the Stage 1 fitted government curve, treat it as a discount curve, and compute what OIS par rates it implies. Compare to the actual quotes. The differences are the asset swap spread curve, term by term.

6. **Turn it on real data.** Find a broker run or an exchange settlement file for EUR OIS — Eurex publishes settlement prices for its OIS futures, and some clearing houses publish end-of-day curves. Replace `data/eur_ois_sample.csv` and see whether the shape you get matches what the financial press says the market is pricing for the ECB.

7. **Two-curve world.** Everything here assumed projection curve = discount curve, which is true for €STR OIS and false for 3M Euribor swaps. Read up on multi-curve bootstrapping, then extend `bootstrap.py` to build a 3M Euribor projection curve discounted on the €STR curve you just built. This is the natural next project, and it is where the toy becomes real.